##### Copyright 2025 Perceptron AI.

In [ ]:
# Licensed under the MIT License (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://opensource.org/licenses/MIT
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Capability — In-context learning (Video)
Show Isaac an example image of an item, then ask it to find that item in a fresh video and return a clip grounding the answer.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/in-context-learning-video.ipynb)

## Install dependencies

In [ ]:
%pip install --upgrade perceptron --quiet

## Download the example image and query video

In [ ]:
ASSET_BASE = "https://raw.githubusercontent.com/perceptron-ai-inc/perceptron/main/cookbook/_shared/assets/capabilities/in-context-learning-video"

!curl -L -so mini_wheats.jpeg {ASSET_BASE}/mini_wheats.jpeg
!curl -L -so cereal_short.mp4 {ASSET_BASE}/cereal_short.mp4

## Configure the Perceptron client

In [ ]:
import os

from perceptron import configure, image, perceive, text, video

api_key = os.getenv("PERCEPTRON_API_KEY", "<your Perceptron API key>")
if not api_key or api_key.startswith("<"):
    raise RuntimeError("Set PERCEPTRON_API_KEY or replace the placeholder in this cell.")

configure(
    provider="perceptron",
    model="isaac-0.3-max",
    api_key=api_key,
)

EXAMPLE_IMAGE = "mini_wheats.jpeg"
QUERY_VIDEO = "cereal_short.mp4"

## Build the in-context sequence and ask
The sequence threads a single example image with a brief intent statement before the query video. Reasoning is on; `expects="clip"` parses the model's `<clip>` tags into structured timestamps.

In [ ]:
@perceive(reasoning=True, expects="clip")
def check_inventory(example_image_path: str, query_video_path: str):
    return (
        image(example_image_path)
        + text("I need to check inventory on this item.")
        + video(query_video_path)
        + text("Is it in stock? Return a clip to justify your answer. Use the <clip> tag to specify clips.")
    )

result = check_inventory(EXAMPLE_IMAGE, QUERY_VIDEO)

print("--- Reasoning ---")
print(result.reasoning or "(none)")
print("\n--- Answer ---")
print(result.text)

## Inspect the returned clips

In [ ]:
clips = result.clips or []
if not clips:
    print("No clips returned - the model may have answered without grounding to a moment.")
else:
    print(f"Returned {len(clips)} clip(s):")
    for idx, clip in enumerate(clips, start=1):
        ts = clip.timestamp
        if ts.until is None:
            window = f"moment at {ts.at:.2f}s"
        else:
            window = f"{ts.at:.2f}s - {ts.until:.2f}s"
        label = clip.mention or "(no mention)"
        print(f"  Clip {idx}: {window} - {label}")

## Conclusion & next steps
- Add additional example shots (more images, or even short reference clips) to specify a richer concept.
- Adjust the intent text to express what kind of grounding you want back ("return a clip when the violation occurs," "return a clip showing the assembly step").
- Pair with [Video Clipping](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/isaac-0.3-max/video-clipping.ipynb) for non-ICL temporal grounding, or with [In-context learning (Image)](https://github.com/perceptron-ai-inc/perceptron/blob/main/cookbook/recipes/capabilities/in-context-learning/in-context-learning.ipynb) when the query is also an image.